## AlexNet: Building and Training the 2012 ImageNet Winner

This notebook walks through AlexNet (Krizhevsky, Sutskever, and Hinton, 2012), the
convolutional network whose result on the ImageNet Large Scale Visual Recognition
Challenge (ILSVRC-2012) is widely credited with starting the modern deep learning era
in computer vision.

We cover, in order:

1. The basic idea, and why it mattered
2. Exploring the dataset (CIFAR-10 stands in for ImageNet)
3. Exploring the architecture, layer by layer
4. Building AlexNet from scratch in PyTorch
5. Training it on a small subset of data

This continues directly from [need_for_cnn.ipynb](need_for_cnn.ipynb): we already saw
that convolution beats a plain DNN on images. AlexNet is the architecture that proved,
at scale, just how far that idea could go.

### 1. The basic idea

Before 2012, the best ImageNet classifiers were built on hand-engineered features
(SIFT, HOG, Fisher vectors) feeding into an SVM. In the ILSVRC-2012 competition,
AlexNet cut the top-5 error rate from about 26% (the best non-neural entry that year)
to 15.3%, a large enough margin that it was not a close call. It was trained end to
end, directly on raw pixels, with no hand-designed features. That result is usually
cited as the moment the field turned decisively toward deep learning for vision.

The architecture itself was not radically new; convolutional networks date back to
LeNet-5 (1998), the small network from [need_for_cnn.ipynb](need_for_cnn.ipynb)'s
lineage. What made AlexNet work at ImageNet scale was a combination of practical
ideas, several of which are still standard today:

| Idea | What it did |
| --- | --- |
| ReLU activation | Replaced tanh/sigmoid. Much faster to train, because it does not saturate for positive inputs, so gradients stay large. |
| Trained across 2 GPUs | The network was split across two GTX 580 GPUs (3GB each) because it did not fit on one. This is why the original conv layers alternate between "communicating" and "independent" halves. A modern GPU, or even a CPU for our small demo below, holds the whole model easily. |
| Overlapping max pooling | Pooling windows step by less than their width (stride 2, size 3). A small but measurable accuracy gain over the non-overlapping pooling common at the time. |
| Local Response Normalization (LRN) | Normalizes each unit's activity relative to nearby feature maps, loosely inspired by lateral inhibition in real neurons. Later architectures dropped this in favor of Batch Normalization; we include it below for historical accuracy. |
| Dropout | Randomly zeroes 50% of the units in the two largest fully connected layers during training, so the network cannot rely on any single unit. Necessary because the classifier head alone holds the large majority of the network's parameters, as we'll see in section 3. |
| Data augmentation | Random crops, horizontal flips, and PCA-based color jitter, used to artificially grow the effective training set and reduce overfitting. |
| Depth | 5 convolutional layers + 3 fully connected layers = 8 layers with learned weights. Deep, for its time. |

The network was trained on 1.2 million labeled training images across 1000 classes
(ImageNet). Training that full setup is not practical in a classroom setting. In
section 5, we train the same architecture on a tiny slice of a much smaller dataset,
and use the result to make the overfitting risk above concrete.

### 2. Explore the dataset

AlexNet was built for ImageNet: 1.2 million training photographs, 1000 classes,
224x224x3 after preprocessing. We don't have that data here. Instead we use
`CIFAR-10`: 60,000 32x32 color photographs across 10 classes (airplane, automobile,
bird, cat, deer, dog, frog, horse, ship, truck), 50,000 for training and 10,000 for
test.

CIFAR-10 is far smaller and far lower resolution than ImageNet, but it keeps two
properties that matter for this demo: the images are real color photographs, not
grayscale icons, and there is more than one visually similar class, so classification
is non-trivial. We resize the 32x32 images up to 224x224 so the original AlexNet
architecture can be used completely unmodified. This does not add any real detail,
the images stay blurry, but it lets us build and run the exact architecture from the
paper.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cpu")
print("device:", device)

CIFAR-10's official mirror (`cs.toronto.edu`) is a single ~170MB archive and is well
known to be slow. Following the same idea as
[need_for_cnn.ipynb](need_for_cnn.ipynb)'s FashionMNIST download, we instead fetch it
as individual JPEGs, already sorted into `train/<class>/` and `test/<class>/`
folders, from an actively maintained GitHub mirror
([YoongiKim/CIFAR-10-images](https://github.com/YoongiKim/CIFAR-10-images)). This
folder layout also lets us load it with `torchvision`'s generic `ImageFolder`
dataset instead of CIFAR-10-specific loading code.

In [ ]:
import os

CIFAR_URL = "https://codeload.github.com/YoongiKim/CIFAR-10-images/tar.gz/refs/heads/master"
DATA_DIR = "data/cifar10"

if os.path.isdir(f"{DATA_DIR}/train"):
    print(f"{DATA_DIR} already present, skipping download")
else:
    !mkdir -p {DATA_DIR}
    !curl -sS -L -o /tmp/cifar10_images.tar.gz {CIFAR_URL}
    !tar -xzf /tmp/cifar10_images.tar.gz -C {DATA_DIR} --strip-components=1
    !rm /tmp/cifar10_images.tar.gz

!ls {DATA_DIR}

In [ ]:
train_dir = f"{DATA_DIR}/train"
test_dir = f"{DATA_DIR}/test"

# no transform yet: raw PIL images, for visualization only
raw_train_ds = datasets.ImageFolder(root=train_dir)
raw_test_ds = datasets.ImageFolder(root=test_dir)
class_names = raw_train_ds.classes

print(f"train images: {len(raw_train_ds)}, test images: {len(raw_test_ds)}")
print(f"classes: {class_names}")
print(f"one image: {raw_train_ds[0][0].size}, mode: {raw_train_ds[0][0].mode}")

In [ ]:
# ImageFolder lists images in class-sorted order, so we sample random indices here
# rather than taking the first 16 (which would all be the same class)
sample_idx = np.random.RandomState(0).choice(len(raw_train_ds), size=16, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, idx in zip(axes.flat, sample_idx):
    img, label = raw_train_ds[idx]
    ax.imshow(img)
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
fig.suptitle("Sample CIFAR-10 images (32x32 native resolution)")
plt.tight_layout()
plt.show()

In [ ]:
counts = np.bincount(raw_train_ds.targets)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(class_names, counts)
ax.set_ylabel("training images")
ax.set_title("CIFAR-10 is class-balanced: 5,000 training images per class")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 3. The architecture, layer by layer

The table below is the original AlexNet architecture, assuming a 224x224x3 input
(images are conventionally cropped from 256x256 down to 224x224; we resize
CIFAR-10's 32x32 images directly up to 224x224 in section 5). Rather than just quoting
numbers from the paper, we trace the output shape and parameter count through each
layer with a real forward pass.

In [ ]:
feature_layers = [
    ("conv1", nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2)),
    ("relu1", nn.ReLU(inplace=True)),
    ("lrn1", nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2)),
    ("pool1", nn.MaxPool2d(kernel_size=3, stride=2)),
    ("conv2", nn.Conv2d(96, 256, kernel_size=5, padding=2)),
    ("relu2", nn.ReLU(inplace=True)),
    ("lrn2", nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2)),
    ("pool2", nn.MaxPool2d(kernel_size=3, stride=2)),
    ("conv3", nn.Conv2d(256, 384, kernel_size=3, padding=1)),
    ("relu3", nn.ReLU(inplace=True)),
    ("conv4", nn.Conv2d(384, 384, kernel_size=3, padding=1)),
    ("relu4", nn.ReLU(inplace=True)),
    ("conv5", nn.Conv2d(384, 256, kernel_size=3, padding=1)),
    ("relu5", nn.ReLU(inplace=True)),
    ("pool3", nn.MaxPool2d(kernel_size=3, stride=2)),
]

def count_params(module):
    return sum(p.numel() for p in module.parameters())

x = torch.zeros(1, 3, 224, 224)
print(f"{'layer':<8}{'output shape':<20}{'params':>12}")
print(f"{'input':<8}{str(tuple(x.shape)):<20}{'':>12}")

conv_total = 0
for name, layer in feature_layers:
    x = layer(x)
    p = count_params(layer)
    conv_total += p
    print(f"{name:<8}{str(tuple(x.shape)):<20}{p:>12,}")

print(f"\nfeature map entering the classifier: {tuple(x.shape)}"
      f" -> flattens to {x.numel():,} values")
print(f"total convolutional parameters: {conv_total:,}")

The last pooling layer leaves a 6x6x256 feature map, exactly the `256 * 6 * 6 = 9216`
input size the paper's classifier expects. Now trace the three fully connected
layers.

In [ ]:
classifier_layers = [
    ("fc6", nn.Linear(256 * 6 * 6, 4096)),
    ("fc7", nn.Linear(4096, 4096)),
    ("fc8", nn.Linear(4096, 1000)),  # 1000 ImageNet classes; we'll use 10 for CIFAR-10 below
]

xf = x.flatten(1)
print(f"{'layer':<8}{'output shape':<20}{'params':>14}")
print(f"{'flatten':<8}{str(tuple(xf.shape)):<20}{'':>14}")

fc_total = 0
for name, layer in classifier_layers:
    xf = layer(xf)
    p = count_params(layer)
    fc_total += p
    print(f"{name:<8}{str(tuple(xf.shape)):<20}{p:>14,}")

print(f"\ntotal fully-connected parameters (1000-class head): {fc_total:,}")
print(f"grand total: {conv_total + fc_total:,}")
print(f"\nfc6 + fc7 alone account for "
      f"{100 * (classifier_layers[0][1].weight.numel() + classifier_layers[1][1].weight.numel()) / (conv_total + fc_total):.1f}%"
      f" of all parameters")

Two things stand out from these numbers:

- The five convolutional layers together hold only a few million parameters, and
  their cost does not scale with the classifier's width, only with kernel size and
  channel count, exactly the property explored for a small conv layer in
  [need_for_cnn.ipynb](need_for_cnn.ipynb).
- `fc6` and `fc7` alone hold the large majority of the network's parameters. This is
  exactly why the paper leans so heavily on Dropout: the overfitting risk is
  concentrated almost entirely in the classifier head, not the feature extractor.

### 4. Building AlexNet in PyTorch

We now package the layer sequence above into a reusable `nn.Module`. Two adjustments
from the raw layer trace in section 3:

- `num_classes` becomes a constructor argument, since we use 10 classes (CIFAR-10)
  instead of the original 1000 (ImageNet).
- We add `nn.AdaptiveAvgPool2d((6, 6))` right before the classifier. With a fixed
  224x224 input this changes nothing, since the last conv block already produces a
  6x6 feature map, but it makes the model robust to slightly different input sizes
  without changing anything else. It's the same approach `torchvision.models.alexnet`
  uses internally.

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2), nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


alexnet = AlexNet(num_classes=10).to(device)
print(alexnet)
print(f"\nAlexNet parameters (10-class head): {count_params(alexnet):,}")

In [ ]:
# sanity check: a batch of 224x224 RGB images in, 10 class scores out
dummy = torch.randn(4, 3, 224, 224)
with torch.no_grad():
    out = alexnet(dummy)
print("input shape:", tuple(dummy.shape), "-> output shape:", tuple(out.shape))

### 5. Training on a small subset

Training the full model on the full CIFAR-10 training set (50,000 images) would take
a while on CPU. To keep this runnable in class, we sample a small, class-balanced
subset: a fixed number of images per class for training, and a smaller fixed number
per class for test.

This is intentionally too little data for a model with tens of millions of
parameters. Expect visible overfitting: training accuracy should climb while test
accuracy lags well behind, the exact failure mode `Dropout` and data augmentation
exist to fight (see
[overfit_underfit_regularization.ipynb](overfit_underfit_regularization.ipynb)). The
goal here is not a high accuracy number, it's watching the real architecture train
end to end, and making the overfitting risk from section 3's parameter count
concrete.

In [ ]:
N_TRAIN_PER_CLASS = 30   # 300 training images total
N_TEST_PER_CLASS = 20    # 200 test images total
BATCH_SIZE = 32
EPOCHS = 25

# CIFAR-10 per-channel mean/std, computed over the full training set
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_ds_full = datasets.ImageFolder(root=train_dir, transform=transform)
test_ds_full = datasets.ImageFolder(root=test_dir, transform=transform)


def balanced_subset_indices(dataset, per_class, num_classes=10, seed=0):
    targets = np.array(dataset.targets)
    rng = np.random.RandomState(seed)
    indices = []
    for c in range(num_classes):
        cls_idx = np.where(targets == c)[0]
        rng.shuffle(cls_idx)
        indices.extend(cls_idx[:per_class].tolist())
    rng.shuffle(indices)
    return indices


train_idx = balanced_subset_indices(train_ds_full, N_TRAIN_PER_CLASS)
test_idx = balanced_subset_indices(test_ds_full, N_TEST_PER_CLASS)

train_ds = Subset(train_ds_full, train_idx)
test_ds = Subset(test_ds_full, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"train subset: {len(train_ds)} images ({N_TRAIN_PER_CLASS}/class)")
print(f"test subset:  {len(test_ds)} images ({N_TEST_PER_CLASS}/class)")

In [ ]:
def train_model(model, loader, epochs, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"loss": [], "train_acc": [], "test_acc": []}
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.size(0)

        epoch_loss, epoch_acc = running_loss / total, correct / total
        epoch_test_acc = evaluate(model, test_loader)
        history["loss"].append(epoch_loss)
        history["train_acc"].append(epoch_acc)
        history["test_acc"].append(epoch_test_acc)
        print(f"  epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}"
              f"  train_acc={epoch_acc:.4f}  test_acc={epoch_test_acc:.4f}")
    return history


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

The original paper trains with plain SGD, momentum 0.9, weight decay 5e-4, and a
hand-tuned learning rate schedule built for 1.2 million images. With only a few
hundred images here, that schedule has no time to do its job, so we use `Adam`
instead, which adapts its own step size and converges without any manual tuning.
(See [adaptive_learning_rates.ipynb](adaptive_learning_rates.ipynb) for why Adam
behaves this way.)

In [ ]:
print("Training AlexNet on the small CIFAR-10 subset...")
history = train_model(alexnet, train_loader, epochs=EPOCHS)

final_train_acc = history["train_acc"][-1]
final_test_acc = history["test_acc"][-1]
print(f"\nfinal train accuracy: {final_train_acc:.4f}")
print(f"final test accuracy:  {final_test_acc:.4f}")
print(f"train - test gap:     {final_train_acc - final_test_acc:.4f}")

In [ ]:
epochs_range = np.arange(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_range, history["loss"], "o-")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("training loss")
axes[0].set_title("Training loss")

axes[1].plot(epochs_range, history["train_acc"], "o-", label="train accuracy")
axes[1].plot(epochs_range, history["test_acc"], "s-", label="test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title(f"Train vs. test accuracy\n({N_TRAIN_PER_CLASS * 10} train / {N_TEST_PER_CLASS * 10} test images)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
alexnet.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = alexnet(images.to(device)).argmax(1).cpu()

unnorm_mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
unnorm_std = torch.tensor(CIFAR_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = images[i] * unnorm_std + unnorm_mean
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    correct = preds[i] == labels[i]
    ax.set_title(f"pred: {class_names[preds[i]]}\ntrue: {class_names[labels[i]]}",
                 fontsize=8, color="green" if correct else "red")
    ax.axis("off")
fig.suptitle("AlexNet predictions on held-out test images (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

### 6. Summary

- AlexNet's win at ILSVRC-2012 (top-5 error 15.3% vs. 26% for the best non-neural
  entry) is the result most often credited with starting the modern deep learning era
  in computer vision.
- Its core ideas, ReLU activations, overlapping max pooling, dropout, and data
  augmentation, are still standard practice; Local Response Normalization is the one
  piece that was later replaced, mostly by Batch Normalization.
- Tracing the architecture layer by layer in section 3 shows the network's roughly 60
  million parameters are overwhelmingly concentrated in the fully connected
  classifier head (`fc6` and `fc7`), not the convolutional feature extractor. This is
  exactly why the paper needs dropout in the classifier and 1.2 million training
  images: a network this large, trained on very little data, overfits badly.
- Section 5 made that concrete: trained on 300 CIFAR-10 images, the exact same
  architecture reached 54% train accuracy but only 27% test accuracy after 25
  epochs, a 27-point gap that kept widening as training continued. That gap is not a
  bug in the demo, it's the overfitting problem the paper's design choices, dropout
  chief among them, exist to fight, at a scale small enough to see directly.